<a href="https://colab.research.google.com/github/Mahammad-Haroon/Data-Analysis-Projects/blob/main/Pandas_Preprocessing_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The Pipeline Imperative

## Logistics Dataset Preprocessing Pipeline

This notebook demonstrates a mathematically ordered preprocessing pipeline consisting of:

1. Deduplication using `order_id`
2. Median imputation of missing `delivery_days`
3. Outlier clipping of `weight_kg`
4. Min-Max normalization of `distance_km` and `weight_kg`

The objective is to create a clean, reproducible dataset and verify the statistical effect of each preprocessing step.

# Answer 1: Pipeline Code & Annotation

The preprocessing operations are applied sequentially because each operation produces the input required by the next operation.

In [1]:
import pandas as pd
import numpy as np

# Step 0 - Load raw, messy logistics dataset
data = {
    'order_id':       [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 3],
    'weight_kg':      [2.1, 0.8, 1.5, 95.0, 3.2, 0.5, 4.1, 2.9, 1.1, 3.7, 0.6, 1.5],
    'distance_km':    [120, 340, 85, 210, 430, 95, 310, 175, 260, 400, 145, 85],
    'delivery_days':  [2, 5, None, 3, 7, 1, 4, None, 3, 6, 2, 3]
}

df = pd.DataFrame(data)

print("=== RAW INPUT SHAPE ===")
print(df.shape)

print("\n=== RAW DATA ===")
print(df)

=== RAW INPUT SHAPE ===
(12, 4)

=== RAW DATA ===
    order_id  weight_kg  distance_km  delivery_days
0          1        2.1          120            2.0
1          2        0.8          340            5.0
2          3        1.5           85            NaN
3          4       95.0          210            3.0
4          5        3.2          430            7.0
5          6        0.5           95            1.0
6          7        4.1          310            4.0
7          8        2.9          175            NaN
8          9        1.1          260            3.0
9         10        3.7          400            6.0
10        11        0.6          145            2.0
11         3        1.5           85            3.0


### Observation

The raw dataset contains 12 rows and 4 columns. The `order_id` value 3 appears twice, indicating one duplicate record. The `delivery_days` column contains two missing values, and `weight_kg` contains an extreme value of 95.0 kg that exceeds the valid domain of 0.1–30.0 kg.

In [2]:
# STEP 1: DEDUPLICATION
# Mathematical logic:
# Each order_id should uniquely identify an order.
# duplicated(order_id) identifies repeated records.
# drop_duplicates() keeps the first occurrence and removes later duplicates.
# This must happen first so duplicate observations do not influence
# later statistics such as the median or scaling parameters.

df = df.drop_duplicates(subset='order_id', keep='first').copy()

print("=== AFTER DEDUPLICATION ===")
print(df)

print("\nShape:", df.shape)

=== AFTER DEDUPLICATION ===
    order_id  weight_kg  distance_km  delivery_days
0          1        2.1          120            2.0
1          2        0.8          340            5.0
2          3        1.5           85            NaN
3          4       95.0          210            3.0
4          5        3.2          430            7.0
5          6        0.5           95            1.0
6          7        4.1          310            4.0
7          8        2.9          175            NaN
8          9        1.1          260            3.0
9         10        3.7          400            6.0
10        11        0.6          145            2.0

Shape: (11, 4)


### Observation

The dataset decreased from 12 to 11 rows because the second occurrence of `order_id = 3` was removed. The first occurrence was retained, ensuring that each order is represented only once before calculating subsequent statistics.

In [3]:
# STEP 2: NULL-HANDLING USING MEDIAN IMPUTATION
# Mathematical logic:
# The median is the middle value after sorting the non-null observations.
# We calculate the median using only the available delivery_days values
# and replace each missing value with that median.
# This step is performed after deduplication so duplicate records cannot
# affect the calculated median.

median_delivery_days = df['delivery_days'].median()

df['delivery_days'] = df['delivery_days'].fillna(median_delivery_days)

print("Median used for imputation:", median_delivery_days)

print("\n=== AFTER MEDIAN IMPUTATION ===")
print(df)

print("\nMissing delivery_days:",
      df['delivery_days'].isna().sum())

Median used for imputation: 3.0

=== AFTER MEDIAN IMPUTATION ===
    order_id  weight_kg  distance_km  delivery_days
0          1        2.1          120            2.0
1          2        0.8          340            5.0
2          3        1.5           85            3.0
3          4       95.0          210            3.0
4          5        3.2          430            7.0
5          6        0.5           95            1.0
6          7        4.1          310            4.0
7          8        2.9          175            3.0
8          9        1.1          260            3.0
9         10        3.7          400            6.0
10        11        0.6          145            2.0

Missing delivery_days: 0


### Observation

The median of the available `delivery_days` values is 3.0 days. Both missing values were replaced with 3.0, reducing the number of missing values in this column from 2 to 0 without being strongly affected by extreme observations.

In [4]:
# STEP 3: OUTLIER CLIPPING
# Mathematical logic:
# Clipping projects every weight value into the valid domain [0.1, 30.0].
# Values below 0.1 become 0.1, while values above 30.0 become 30.0.
# The 95.0 kg observation is therefore replaced with 30.0 kg.
# This must happen before Min-Max normalization so the extreme value
# does not distort the normalization range.

df['weight_kg'] = df['weight_kg'].clip(lower=0.1, upper=30.0)

print("=== AFTER OUTLIER CLIPPING ===")
print(df)

print("\nMinimum weight:", df['weight_kg'].min())
print("Maximum weight:", df['weight_kg'].max())

=== AFTER OUTLIER CLIPPING ===
    order_id  weight_kg  distance_km  delivery_days
0          1        2.1          120            2.0
1          2        0.8          340            5.0
2          3        1.5           85            3.0
3          4       30.0          210            3.0
4          5        3.2          430            7.0
5          6        0.5           95            1.0
6          7        4.1          310            4.0
7          8        2.9          175            3.0
8          9        1.1          260            3.0
9         10        3.7          400            6.0
10        11        0.6          145            2.0

Minimum weight: 0.5
Maximum weight: 30.0


### Observation

The extreme `weight_kg` value of 95.0 was clipped to the valid upper boundary of 30.0 kg. The resulting weight values now satisfy the required domain constraint, with the maximum value equal to 30.0.

In [5]:
# STEP 4: MIN-MAX NORMALIZATION
# Mathematical logic:
# Min-Max scaling transforms each value x using:
#
# x_scaled = (x - x_min) / (x_max - x_min)
#
# This maps the minimum observed value to 0 and the maximum observed
# value to 1.
#
# Scaling is performed AFTER clipping so the 95.0 kg outlier does not
# become the maximum used in the normalization denominator.

for column in ['distance_km', 'weight_kg']:
    min_value = df[column].min()
    max_value = df[column].max()

    df[column] = (
        (df[column] - min_value)
        / (max_value - min_value)
    )

print("=== FINAL PREPROCESSED DATA ===")
print(df)

=== FINAL PREPROCESSED DATA ===
    order_id  weight_kg  distance_km  delivery_days
0          1   0.054237     0.101449            2.0
1          2   0.010169     0.739130            5.0
2          3   0.033898     0.000000            3.0
3          4   1.000000     0.362319            3.0
4          5   0.091525     1.000000            7.0
5          6   0.000000     0.028986            1.0
6          7   0.122034     0.652174            4.0
7          8   0.081356     0.260870            3.0
8          9   0.020339     0.507246            3.0
9         10   0.108475     0.913043            6.0
10        11   0.003390     0.173913            2.0
